# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant Dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore available record sets, fields, and columns by @id
print('Available record sets:')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}, description: {rs.description}")

# For illustration: print out all fields and columns for each record set
for rs in record_sets:
    print(f"\nRecordSet: {rs.id}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}, description: {field.description}")
    print('  Columns:')
    for column in rs.columns:
        print(f"    - @id: {column.id}, name: {column.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print('Available RecordSet @ids:', record_set_ids)

# For this notebook, we use the first record set by default for demonstration:
if record_set_ids:
    chosen_rs_id = record_set_ids[0]
    print(f"\nExtracting records from RecordSet: {chosen_rs_id}\n")
    # Load records
    records = list(dataset.records(record_set=chosen_rs_id))
    df = pd.DataFrame(records)
    # Show the fields (columns)
    print('Fields (@id) of chosen record set:')
    print(list(df.columns))
    display_cols = df.columns.tolist()[:10]  # Limit to 10 columns for output
    print(f"First {len(display_cols)} columns: {display_cols}")
    df.head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data, or grouping by attributes.

In [ ]:
import numpy as np
# Try to automatically detect a numeric field for demo

numeric_fields = []
for col in df.columns:
    try:
        # Test if column can be converted to float
        df[col+'_float'] = pd.to_numeric(df[col], errors='coerce')
        if df[col+'_float'].notnull().sum() > 0:
            numeric_fields.append(col)
    except Exception:
        pass
if not numeric_fields:
    print('No apparent numeric fields found.')
else:
    # Use the first numeric field
    numeric_field = numeric_fields[0]
    print(f'Using numeric field for EDA: {numeric_field}')
    # Filter on a threshold: e.g., mean
    df[numeric_field+'_float'] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field+'_float'].mean()
    filtered_df = df[df[numeric_field+'_float'] > threshold]
    print(f"Records with {numeric_field} > {threshold:.2f} : {len(filtered_df)} rows")
    # Z-score normalization
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field+'_float'] - filtered_df[numeric_field+'_float'].mean()
    ) / filtered_df[numeric_field+'_float'].std()
    print(filtered_df[[numeric_field, numeric_field+'_float', numeric_field+'_normalized']].head())

    # Try grouping by a categorical field
    group_field = None
    for c in df.columns:
        if c != numeric_field and df[c].nunique() < min(10, len(df)//4):
            group_field = c
            break
    if group_field:
        print(f"Grouping by categorical field: {group_field}")
        grouped = (
            filtered_df.groupby(group_field)[numeric_field+'_float'].mean()
            .reset_index().sort_values(numeric_field+'_float', ascending=False)
        )
        print(grouped)
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field+'_float'].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group (if group_field found)
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field+'_float', data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

- We reviewed the dataset structure using the Croissant schema, referencing all entities by their `@id`.
- We extracted tabular records from the record set(s) and explored the available fields.
- An initial EDA examined the distribution of a numeric field, normalized it, and, if possible, grouped by a categorical field to reveal trends.
- Visualizations compared feature distributions and grouped statistics.

For deeper domain analysis, users are encouraged to consult the dataset schema for precise field definitions, and to design additional analytic workflows relevant to the biological or clinical context.